In [ ]:
import h5py
import cv2
import numpy as np
from pathlib import Path
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

In [ ]:
def post_processing(h5_path, output_folder, min_distance):
    h5_path = Path(h5_path)
    output_folder = Path(output_folder)

    h5_filename = h5_path.stem
    output_path = output_folder / h5_filename

    output_path.mkdir(parents=True, exist_ok=True) #parents=True隨路徑遞歸建立目錄 #exist_ok=True存在即跳過

    report_lines = []

    with h5py.File(h5_path, 'r') as f:
        for img_name in f.keys(): #f.keys()釣出f裡所有group名組成list
            print(f"processing image: {img_name}")

            group = f[img_name] #a[目標][指定讀取範圍, 進入記憶體]
            raw_img = group['raw_image'][:]
            prob_map = group['probability_map'][:]
            h, w = prob_map.shape #raw_img.shape = (H, W, 3), prob_map.shape = (H, W)

            thresh = prob_map >= 0.5
            distance = ndi.distance_transform_edt(thresh)
                #Euclidean Distance Transform, EDT #必須輸入boolean matrix, False=背景像素, True=前景像素
                #算每一個True與最近的False的EDT距離, 存成matrix
            coords = peak_local_max(distance, min_distance=min_distance, labels=thresh)
                #peak_local_max()在一維或多維矩陣中, 尋找局部區域內的極大值peaks
                #peak_local_max(目標矩陣, 最短距離threshold, 區域限制) #最短距離threshold 加大減少沾黏, 縮小減少過度計算
                #區域限制 根據與目標矩陣同大小的boolean matrix, True即為範圍
            mask = np.zeros(distance.shape, dtype=bool)
            mask[tuple(coords.T)] = True
                #將細胞中心座標寫成boolean matrix
                #mask[] numpy多維索引賦值, 需求格式為(array([y1, y2, y3]), array([x1, x2, x3]))
                #coords原格式[[y1, x1], [y2, x2], ...] >>> .T後[[y1, y2, y3, ...], [x1, x2, x3, ...]]
                #tuple()元祖化(array([y1, y2, y3, ...]), array([x1, x2, x3, ...]))
            markers, _ = ndi.label(mask) #a, b = ndi.label(矩陣) 將矩陣中兩個相連的非0(or True)元素視為同一元素, 輸出a為元素矩陣map(標記編號), 輸出b為元素數量
            labels = watershed(-distance, markers, mask=thresh)
                #分水領算法Marker-controlled Watershed Algorithm #watershed(地形圖, 谷地中心, 邊界)
                #地形圖 distance加負號為數字做反轉, 原正數時中心最高, 需要中心最低
                #谷地中心 標記帶有編號的獨立中心 #邊界 用boolean matrix的True標記邊界

            complete_count = 0
            incomplete_count = 0
            overlay_img = raw_img.copy()

            for label_id in np.unique(labels): #np.unique()找出矩陣中所有出現過的不重複數值, 並由小到大排列, 格式[0, 1, 2, 3, ..., N]
                if label_id == 0: continue
                cell_mask = (labels == label_id) #每一次for loop為該編號細胞生成一張boolean matrix
                y_coords, x_coords = np.where(cell_mask) #np.where()用y, x回傳所有非0數值的座標, 格式y_coords = array([]), x_coords = array([])